## Imports and paths

In [7]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

## Split Preparation

In [8]:
# The Folder Path
INPUT_PATH = Path("../Cleaned Data/cleaned_data_preprocessing_renamed.csv")
OUTPUT_DIR = Path("../Cleaned Data/Splits")


TEST_SIZE = 0.20
RANDOM_STATE = 1948883  # My Student ID


# Columns
REQUIRED_COLUMNS = ["Record ID","Survey Completion Date","If Within Mask Mandate Period","Face Mask Wearing Score","Face Mask Wearing Binary","Overall Protective Behavior Score","Overall Protective Behavior Binary",]

## Helper functions

In [9]:
# Check required columns
def validate_input_columns(df: pd.DataFrame) -> None:
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError("Error1")



# Convert the mask mandate period column to integer format
def normalize_period_column(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()
    period_col = "If Within Mask Mandate Period"
    df[period_col] = df[period_col].astype(int)
    valid_values = set(df[period_col].dropna().unique().tolist())
    if not valid_values.issubset({0, 1}):
        raise ValueError("Error2")
    
    return df


# Converts the target variable to binary 0/1
def normalize_binary_target(y: pd.Series) -> pd.Series:
    y_clean = y.copy()
    try:
        y_num = pd.to_numeric(y_clean, errors="raise").astype(int)
        valid_values = set(y_num.dropna().unique().tolist())
        if valid_values.issubset({0, 1}):
            return y_num.rename("y")
    except Exception:
        pass

    mapping = {"0": 0,"1": 1,"false": 0,"true": 1,"no": 0,"yes": 1,}
    y_str = y_clean.astype(str).str.strip().str.lower()
    if not set(y_str.unique()).issubset(set(mapping.keys())):
        raise ValueError("Error3")

    return y_str.map(mapping).astype(int).rename("y")



# Split to training sets and test sets
def split_task_data(X: pd.DataFrame, y: pd.Series):

    class_counts = y.value_counts(dropna=False).to_dict()
    use_stratify = (y.nunique() == 2 and min(class_counts.values()) >= 2)
    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=TEST_SIZE,random_state=RANDOM_STATE,stratify=y if use_stratify else None)

    return X_train, X_test, y_train, y_test



# Save
def save_split_files(X_train: pd.DataFrame,X_test: pd.DataFrame,y_train: pd.Series,y_test: pd.Series,task_name: str,output_dir: Path,) -> None:

    output_dir.mkdir(parents=True, exist_ok=True)

    X_train.to_csv(output_dir / f"X_train_{task_name}.csv", index=False)
    X_test.to_csv(output_dir / f"X_test_{task_name}.csv", index=False)

    pd.DataFrame({"y": y_train}).to_csv(output_dir / f"y_train_{task_name}.csv",index=False,)
    pd.DataFrame({"y": y_test}).to_csv(output_dir / f"y_test_{task_name}.csv",index=False,)



# Modeling task
def build_one_task(df_subset: pd.DataFrame,target_col: str,drop_cols: list,task_name: str,output_dir: Path,) -> dict:

    drop_cols = [c for c in drop_cols if c in df_subset.columns]                        # REmove Some columns

    X = df_subset.drop(columns=drop_cols).copy()                                        # X is predictor variables
    y = normalize_binary_target(df_subset[target_col])                                  # y is the binary target variable

    if X.empty:
        raise ValueError(f"Task '{task_name}' produced an empty X matrix.")
    
    X_train, X_test, y_train, y_test = split_task_data(X, y)
    save_split_files(X_train, X_test, y_train, y_test, task_name, output_dir)          
    
    return {"task_name": task_name,"target": target_col,"n_total": len(df_subset),"n_train": len(X_train),"n_test": len(X_test),"n_features": X.shape[1],"positive_rate_total": round(float(y.mean()), 6),"positive_rate_train": round(float(y_train.mean()), 6),"positive_rate_test": round(float(y_test.mean()), 6),}

## Build and save the split datasets

In [10]:
def main():
    if not INPUT_PATH.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_PATH}")


    # Read
    df = pd.read_csv(INPUT_PATH, keep_default_na=False)


    # Check required columns
    validate_input_columns(df)
    df = normalize_period_column(df)
    period_col = "If Within Mask Mandate Period"


    # Split the full dataset into before-mandate and after-mandate subsets.
    before_df = df.loc[df[period_col] == 0].copy()
    after_df = df.loc[df[period_col] == 1].copy()


    if before_df.empty:
        raise ValueError("No rows found for BEFORE mandate data.")
    if after_df.empty:
        raise ValueError("No rows found for AFTER mandate data.")
    summaries = []


    # Common columns removed from all modeling tasks.
    common_drop_cols = ["Record ID","Survey Completion Date","If Within Mask Mandate Period","Face Mask Wearing Score","Face Mask Wearing Binary","Overall Protective Behavior Score","Overall Protective Behavior Binary",]


    # For protective behavior prediction, this extra score is also removed to avoid target leakage.
    protective_extra_drop_cols = common_drop_cols + ["Protective Behavior Score Excluding Mask"]

    # Predict face mask wearing before the mask mandate.
    summaries.append(build_one_task(df_subset=before_df,target_col="Face Mask Wearing Binary",drop_cols=common_drop_cols,task_name="before_mask",output_dir=OUTPUT_DIR))

    # Predict face mask wearing after the mask mandate.
    summaries.append(build_one_task(df_subset=after_df,target_col="Face Mask Wearing Binary",drop_cols=common_drop_cols,task_name="after_mask",output_dir=OUTPUT_DIR,))

    # Predict overall protective behavior before the mask mandate.
    summaries.append(build_one_task(df_subset=before_df,target_col="Overall Protective Behavior Binary",drop_cols=protective_extra_drop_cols,task_name="before_protective",output_dir=OUTPUT_DIR,))

    # Predict overall protective behavior after the mask mandate.
    summaries.append(build_one_task(df_subset=after_df,target_col="Overall Protective Behavior Binary",drop_cols=protective_extra_drop_cols,task_name="after_protective",output_dir=OUTPUT_DIR))

    # Save
    summary_df = pd.DataFrame(summaries)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    summary_df.to_csv(OUTPUT_DIR / "split_summary.csv", index=False)


    print("\nSummary:")
    print(summary_df.to_string(index=False))


if __name__ == "__main__":
    main()


Summary:
        task_name                             target  n_total  n_train  n_test  n_features  positive_rate_total  positive_rate_train  positive_rate_test
      before_mask           Face Mask Wearing Binary    10939     8751    2188          58             0.279550             0.279511            0.279707
       after_mask           Face Mask Wearing Binary    24521    19616    4905          58             0.704580             0.704578            0.704587
before_protective Overall Protective Behavior Binary    10939     8751    2188          57             0.492824             0.492858            0.492687
 after_protective Overall Protective Behavior Binary    24521    19616    4905          57             0.696831             0.696829            0.696840
